In [ ]:
!pip install pandas scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving tmdb_5000_movies.csv to tmdb_5000_movies.csv
Saving tmdb_5000_credits.csv to tmdb_5000_credits (2).csv


In [ ]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

movies = movies.merge(credits, on="title")

In [ ]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [ ]:
movies.dropna(inplace=True)

In [ ]:
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(lambda x: convert(x)[:3])

In [ ]:
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [ ]:
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))

In [ ]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(movies['tags']).toarray()

In [ ]:
similarity = cosine_similarity(vectors)

In [ ]:
def recommend(movie):
    movie_index = movies[movies['title'] == movie].index[0]
    distances = similarity[movie_index]

    movie_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

    for i in movie_list:
        print(movies.iloc[i[0]].title)

In [26]:
recommend("Frozen")

Delgo
Aladdin
The Book of Life
Stardust
Spirit: Stallion of the Cimarron


In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
def recommend_ui(movie):
    try:
        movie_index = movies[movies['title'] == movie].index[0]
        distances = similarity[movie_index]

        movie_list = sorted(
            list(enumerate(distances)),
            reverse=True,
            key=lambda x: x[1]
        )[1:6]

        results = [movies.iloc[i[0]].title for i in movie_list]

        return "\n".join(results)

    except:
        return "❌ Movie not found. Try another name."


with gr.Blocks(theme=gr.themes.Soft(), css="""
#title {text-align: center; font-size: 40px; font-weight: bold;}
#subtitle {text-align: center; color: gray; margin-bottom: 20px;}
.gradio-container {max-width: 100% !important;}
""") as app:

    gr.Markdown("<div id='title'>🎥 Movie Recommendation System</div>")
    gr.Markdown("<div id='subtitle'>Find movies similar to your favorite one</div>")

    with gr.Row():
        with gr.Column():
            movie_input = gr.Textbox(
                label="Enter Movie Name 🎬",
                placeholder="e.g. Avatar"
            )
            btn = gr.Button("✨ Recommend")

        with gr.Column():
            output = gr.Textbox(
                label="Recommended Movies 🍿",
                lines=6
            )

    btn.click(recommend_ui, inputs=movie_input, outputs=output)


app.launch(share=True, inline=False)

/tmp/ipykernel_1320/1823419368.py:21: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css="""


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c73fac7e3c79e607e1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
